In [1]:
import geopandas as gpd
import numpy as np
import os, json, pickle, ast
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
GEOJSON_FOLDER = "/home/group4/project/test_data" #( 5x5)
OUTPUT_FOLDER  = "/home/group4/Transformers_Nathalia/project/Output_data/test"
LABEL_LEVEL    = "l3_species"   # 19 classes — change to "l2_genus" or "l1_leaf_types" if needed
# ────────────────────────────────────────────────────────────────────────────

BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12',
         'EVI', 'EVI2', 'NDVI', 'NDWI', 'SAVI']  # C = 15
N_DATES  = 8
N_PIXELS = 25   # 5x5 patch, flattened

# Actual acquisition dates from your GEE script
DATES = {
    "0": "20220301",
    "1": "20220401",
    "2": "20220501",
    "3": "20220601",
    "4": "20220701",
    "5": "20220801",
    "6": "20220901",
    "7": "20221001",
}

def get_col(band, t):
    return band if t == 0 else f"{band}_{t}"

def parse_cell(val):
    """Parse nested list cell → flat float32 array of length N_PIXELS."""
    if val is None:
        return None
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except:
            return None
    arr = np.array(val, dtype=np.float32).flatten()  # (5,5) → (25,)
    if len(arr) != N_PIXELS:
        return None  # unexpected shape, skip
    return arr

def row_to_tcs(row):
    """
    Convert one GDF row → array (T=8, C=15, S=25).
    Also returns a boolean valid mask of shape (T,).
    """
    out   = np.zeros((N_DATES, len(BANDS), N_PIXELS), dtype=np.float32)
    valid = np.ones(N_DATES, dtype=bool)

    for t in range(N_DATES):
        for c, band in enumerate(BANDS):
            arr = parse_cell(row[get_col(band, t)])
            if arr is None:
                valid[t] = False
                out[t, c, :] = 0.0   # fill missing date with zeros
            else:
                out[t, c, :] = arr

    return out, valid

# ── MAIN BUILD ───────────────────────────────────────────────────────────────
def build_dataset(geojson_folder, output_folder, label_level=LABEL_LEVEL):
    out = Path(output_folder)
    (out / "DATA").mkdir(parents=True, exist_ok=True)
    (out / "META").mkdir(parents=True, exist_ok=True)

    files = sorted([f for f in os.listdir(geojson_folder) if f.endswith(".geojson")])

    # ── Build label map from all files ──────────────────────────────────────
    label_set = set()
    for fname in files:
        gdf = gpd.read_file(os.path.join(geojson_folder, fname))
        label_set.update(gdf[label_level].dropna().unique())
    label_map = {name: i for i, name in enumerate(sorted(label_set))}
    print(f"Found {len(label_map)} classes:")
    for name, idx in label_map.items():
        print(f"  [{idx}] {name}")

    # ── Process each file ───────────────────────────────────────────────────
    labels_dict = {}
    all_arrays  = []
    sample_id   = 0
    skipped     = 0

    for fname in files:
        gdf = gpd.read_file(os.path.join(geojson_folder, fname))
        species = fname.replace(".geojson", "")
        print(f"\n▶ {species}  ({len(gdf)} trees)")

        for _, row in gdf.iterrows():
            label = row.get(label_level)
            if label not in label_map:
                skipped += 1
                continue

            arr, valid = row_to_tcs(row)

            # Skip if more than 2 dates are missing (out of 8)
            if valid.sum() < 6:
                skipped += 1
                continue

            uid = f"sample_{sample_id:06d}"
            np.save(out / "DATA" / f"{uid}.npy", arr)
            labels_dict[uid] = label_map[label]
            all_arrays.append(arr)
            sample_id += 1

        print(f"  → {sample_id} samples saved so far")

    print(f"\n✓ Total samples : {sample_id}")
    print(f"  Skipped        : {skipped}")
    print(f"  Array shape    : T={N_DATES}, C={len(BANDS)}, S={N_PIXELS}")

    # ── labels.json ─────────────────────────────────────────────────────────
    with open(out / "META" / "labels.json", "w") as f:
        json.dump({label_level: labels_dict}, f, indent=2)

    # ── class_map.json (human-readable, for your reference) ─────────────────
    with open(out / "META" / "class_map.json", "w") as f:
        json.dump({v: k for k, v in label_map.items()}, f, indent=2)  # idx → name

    # ── dates.json ──────────────────────────────────────────────────────────
    with open(out / "META" / "dates.json", "w") as f:
        json.dump(DATES, f, indent=2)

    # ── normalisation_values.pkl (channel-wise mean/std) ────────────────────
    big   = np.stack(all_arrays, axis=0)          # (N, T, C, S)
    means = big.mean(axis=(0, 1, 3))              # (C,)
    stds  = big.std(axis=(0, 1, 3))               # (C,)
    stds[stds == 0] = 1e-6
    with open(out / "normalisation_values.pkl", "wb") as f:
        pickle.dump((means, stds), f)

    print(f"✓ Normalisation: means={np.round(means,4)}, stds={np.round(stds,4)}")
    print(f"\n✓ Dataset ready at: {output_folder}")
    print(f"  class_map saved to META/class_map.json — keep this for decoding predictions!")

build_dataset(GEOJSON_FOLDER, OUTPUT_FOLDER)

Found 19 classes:
  [0] alder
  [1] birch
  [2] black pine
  [3] cherry
  [4] douglas fir
  [5] english oak
  [6] european ash
  [7] european beech
  [8] european larch
  [9] japanese larch
  [10] linden
  [11] norway spruce
  [12] poplar
  [13] red oak
  [14] scots pine
  [15] sessile oak
  [16] silver fir
  [17] sycamore maple
  [18] weymouth pine

▶ 5x5_test_data_alder  (193 trees)
  → 193 samples saved so far

▶ 5x5_test_data_birch  (165 trees)
  → 358 samples saved so far

▶ 5x5_test_data_blackpine  (2 trees)
  → 360 samples saved so far

▶ 5x5_test_data_cherry  (14 trees)
  → 374 samples saved so far

▶ 5x5_test_data_douglasfir  (420 trees)
  → 794 samples saved so far

▶ 5x5_test_data_englishoak  (274 trees)
  → 1068 samples saved so far

▶ 5x5_test_data_europeanash  (48 trees)
  → 1116 samples saved so far

▶ 5x5_test_data_europeanbeech  (171 trees)
  → 1287 samples saved so far

▶ 5x5_test_data_europeanlarch  (117 trees)
  → 1404 samples saved so far

▶ 5x5_test_data_japanesel

In [2]:
import geopandas as gpd
import numpy as np
import os, json, pickle, ast
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
GEOJSON_FOLDER = "/home/group4/project/test_data" #( 5x5)
OUTPUT_FOLDER  = "/home/group4/Transformers_Nathalia/project/Output_data/test_spectral"
LABEL_LEVEL    = "l3_species"   # 19 classes — change to "l2_genus" or "l1_leaf_types" if needed
# ────────────────────────────────────────────────────────────────────────────

BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']  # C = 10
N_DATES  = 8
N_PIXELS = 25   # 5x5 patch, flattened

# Actual acquisition dates from your GEE script
DATES = {
    "0": "20220301",
    "1": "20220401",
    "2": "20220501",
    "3": "20220601",
    "4": "20220701",
    "5": "20220801",
    "6": "20220901",
    "7": "20221001",
}

def get_col(band, t):
    return band if t == 0 else f"{band}_{t}"

def parse_cell(val):
    """Parse nested list cell → flat float32 array of length N_PIXELS."""
    if val is None:
        return None
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except:
            return None
    arr = np.array(val, dtype=np.float32).flatten()  # (5,5) → (25,)
    if len(arr) != N_PIXELS:
        return None  # unexpected shape, skip
    return arr

def row_to_tcs(row):
    """
    Convert one GDF row → array (T=8, C=15, S=25).
    Also returns a boolean valid mask of shape (T,).
    """
    out   = np.zeros((N_DATES, len(BANDS), N_PIXELS), dtype=np.float32)
    valid = np.ones(N_DATES, dtype=bool)

    for t in range(N_DATES):
        for c, band in enumerate(BANDS):
            arr = parse_cell(row[get_col(band, t)])
            if arr is None:
                valid[t] = False
                out[t, c, :] = 0.0   # fill missing date with zeros
            else:
                out[t, c, :] = arr

    return out, valid

# ── MAIN BUILD ───────────────────────────────────────────────────────────────
def build_dataset(geojson_folder, output_folder, label_level=LABEL_LEVEL):
    out = Path(output_folder)
    (out / "DATA").mkdir(parents=True, exist_ok=True)
    (out / "META").mkdir(parents=True, exist_ok=True)

    files = sorted([f for f in os.listdir(geojson_folder) if f.endswith(".geojson")])

    # ── Build label map from all files ──────────────────────────────────────
    label_set = set()
    for fname in files:
        gdf = gpd.read_file(os.path.join(geojson_folder, fname))
        label_set.update(gdf[label_level].dropna().unique())
    label_map = {name: i for i, name in enumerate(sorted(label_set))}
    print(f"Found {len(label_map)} classes:")
    for name, idx in label_map.items():
        print(f"  [{idx}] {name}")

    # ── Process each file ───────────────────────────────────────────────────
    labels_dict = {}
    all_arrays  = []
    sample_id   = 0
    skipped     = 0

    for fname in files:
        gdf = gpd.read_file(os.path.join(geojson_folder, fname))
        species = fname.replace(".geojson", "")
        print(f"\n▶ {species}  ({len(gdf)} trees)")

        for _, row in gdf.iterrows():
            label = row.get(label_level)
            if label not in label_map:
                skipped += 1
                continue

            arr, valid = row_to_tcs(row)

            # Skip if more than 2 dates are missing (out of 8)
            if valid.sum() < 6:
                skipped += 1
                continue

            uid = f"sample_{sample_id:06d}"
            np.save(out / "DATA" / f"{uid}.npy", arr)
            labels_dict[uid] = label_map[label]
            all_arrays.append(arr)
            sample_id += 1

        print(f"  → {sample_id} samples saved so far")

    print(f"\n✓ Total samples : {sample_id}")
    print(f"  Skipped        : {skipped}")
    print(f"  Array shape    : T={N_DATES}, C={len(BANDS)}, S={N_PIXELS}")

    # ── labels.json ─────────────────────────────────────────────────────────
    with open(out / "META" / "labels.json", "w") as f:
        json.dump({label_level: labels_dict}, f, indent=2)

    # ── class_map.json (human-readable, for your reference) ─────────────────
    with open(out / "META" / "class_map.json", "w") as f:
        json.dump({v: k for k, v in label_map.items()}, f, indent=2)  # idx → name

    # ── dates.json ──────────────────────────────────────────────────────────
    with open(out / "META" / "dates.json", "w") as f:
        json.dump(DATES, f, indent=2)

    # ── normalisation_values.pkl (channel-wise mean/std) ────────────────────
    big   = np.stack(all_arrays, axis=0)          # (N, T, C, S)
    means = big.mean(axis=(0, 1, 3))              # (C,)
    stds  = big.std(axis=(0, 1, 3))               # (C,)
    stds[stds == 0] = 1e-6
    with open(out / "normalisation_values.pkl", "wb") as f:
        pickle.dump((means, stds), f)

    print(f"✓ Normalisation: means={np.round(means,4)}, stds={np.round(stds,4)}")
    print(f"\n✓ Dataset ready at: {output_folder}")
    print(f"  class_map saved to META/class_map.json — keep this for decoding predictions!")

build_dataset(GEOJSON_FOLDER, OUTPUT_FOLDER)

Found 19 classes:
  [0] alder
  [1] birch
  [2] black pine
  [3] cherry
  [4] douglas fir
  [5] english oak
  [6] european ash
  [7] european beech
  [8] european larch
  [9] japanese larch
  [10] linden
  [11] norway spruce
  [12] poplar
  [13] red oak
  [14] scots pine
  [15] sessile oak
  [16] silver fir
  [17] sycamore maple
  [18] weymouth pine

▶ 5x5_test_data_alder  (193 trees)
  → 193 samples saved so far

▶ 5x5_test_data_birch  (165 trees)
  → 358 samples saved so far

▶ 5x5_test_data_blackpine  (2 trees)
  → 360 samples saved so far

▶ 5x5_test_data_cherry  (14 trees)
  → 374 samples saved so far

▶ 5x5_test_data_douglasfir  (420 trees)
  → 794 samples saved so far

▶ 5x5_test_data_englishoak  (274 trees)
  → 1068 samples saved so far

▶ 5x5_test_data_europeanash  (48 trees)
  → 1116 samples saved so far

▶ 5x5_test_data_europeanbeech  (171 trees)
  → 1287 samples saved so far

▶ 5x5_test_data_europeanlarch  (117 trees)
  → 1404 samples saved so far

▶ 5x5_test_data_japanesel

In [3]:
import geopandas as gpd
import numpy as np
import os, json, pickle, ast
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
GEOJSON_FOLDER = "/home/group4/project/test_data" #( 5x5)
OUTPUT_FOLDER  = "/home/group4/Transformers_Nathalia/project/Output_data/test_vegetation"
LABEL_LEVEL    = "l3_species"   # 19 classes — change to "l2_genus" or "l1_leaf_types" if needed
# ────────────────────────────────────────────────────────────────────────────

BANDS = ['EVI', 'EVI2', 'NDVI', 'NDWI', 'SAVI']  # C = 5
N_DATES  = 8
N_PIXELS = 25   # 5x5 patch, flattened

# Actual acquisition dates from your GEE script
DATES = {
    "0": "20220301",
    "1": "20220401",
    "2": "20220501",
    "3": "20220601",
    "4": "20220701",
    "5": "20220801",
    "6": "20220901",
    "7": "20221001",
}

def get_col(band, t):
    return band if t == 0 else f"{band}_{t}"

def parse_cell(val):
    """Parse nested list cell → flat float32 array of length N_PIXELS."""
    if val is None:
        return None
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except:
            return None
    arr = np.array(val, dtype=np.float32).flatten()  # (5,5) → (25,)
    if len(arr) != N_PIXELS:
        return None  # unexpected shape, skip
    return arr

def row_to_tcs(row):
    """
    Convert one GDF row → array (T=8, C=15, S=25).
    Also returns a boolean valid mask of shape (T,).
    """
    out   = np.zeros((N_DATES, len(BANDS), N_PIXELS), dtype=np.float32)
    valid = np.ones(N_DATES, dtype=bool)

    for t in range(N_DATES):
        for c, band in enumerate(BANDS):
            arr = parse_cell(row[get_col(band, t)])
            if arr is None:
                valid[t] = False
                out[t, c, :] = 0.0   # fill missing date with zeros
            else:
                out[t, c, :] = arr

    return out, valid

# ── MAIN BUILD ───────────────────────────────────────────────────────────────
def build_dataset(geojson_folder, output_folder, label_level=LABEL_LEVEL):
    out = Path(output_folder)
    (out / "DATA").mkdir(parents=True, exist_ok=True)
    (out / "META").mkdir(parents=True, exist_ok=True)

    files = sorted([f for f in os.listdir(geojson_folder) if f.endswith(".geojson")])

    # ── Build label map from all files ──────────────────────────────────────
    label_set = set()
    for fname in files:
        gdf = gpd.read_file(os.path.join(geojson_folder, fname))
        label_set.update(gdf[label_level].dropna().unique())
    label_map = {name: i for i, name in enumerate(sorted(label_set))}
    print(f"Found {len(label_map)} classes:")
    for name, idx in label_map.items():
        print(f"  [{idx}] {name}")

    # ── Process each file ───────────────────────────────────────────────────
    labels_dict = {}
    all_arrays  = []
    sample_id   = 0
    skipped     = 0

    for fname in files:
        gdf = gpd.read_file(os.path.join(geojson_folder, fname))
        species = fname.replace(".geojson", "")
        print(f"\n▶ {species}  ({len(gdf)} trees)")

        for _, row in gdf.iterrows():
            label = row.get(label_level)
            if label not in label_map:
                skipped += 1
                continue

            arr, valid = row_to_tcs(row)

            # Skip if more than 2 dates are missing (out of 8)
            if valid.sum() < 6:
                skipped += 1
                continue

            uid = f"sample_{sample_id:06d}"
            np.save(out / "DATA" / f"{uid}.npy", arr)
            labels_dict[uid] = label_map[label]
            all_arrays.append(arr)
            sample_id += 1

        print(f"  → {sample_id} samples saved so far")

    print(f"\n✓ Total samples : {sample_id}")
    print(f"  Skipped        : {skipped}")
    print(f"  Array shape    : T={N_DATES}, C={len(BANDS)}, S={N_PIXELS}")

    # ── labels.json ─────────────────────────────────────────────────────────
    with open(out / "META" / "labels.json", "w") as f:
        json.dump({label_level: labels_dict}, f, indent=2)

    # ── class_map.json (human-readable, for your reference) ─────────────────
    with open(out / "META" / "class_map.json", "w") as f:
        json.dump({v: k for k, v in label_map.items()}, f, indent=2)  # idx → name

    # ── dates.json ──────────────────────────────────────────────────────────
    with open(out / "META" / "dates.json", "w") as f:
        json.dump(DATES, f, indent=2)

    # ── normalisation_values.pkl (channel-wise mean/std) ────────────────────
    big   = np.stack(all_arrays, axis=0)          # (N, T, C, S)
    means = big.mean(axis=(0, 1, 3))              # (C,)
    stds  = big.std(axis=(0, 1, 3))               # (C,)
    stds[stds == 0] = 1e-6
    with open(out / "normalisation_values.pkl", "wb") as f:
        pickle.dump((means, stds), f)

    print(f"✓ Normalisation: means={np.round(means,4)}, stds={np.round(stds,4)}")
    print(f"\n✓ Dataset ready at: {output_folder}")
    print(f"  class_map saved to META/class_map.json — keep this for decoding predictions!")

build_dataset(GEOJSON_FOLDER, OUTPUT_FOLDER)

Found 19 classes:
  [0] alder
  [1] birch
  [2] black pine
  [3] cherry
  [4] douglas fir
  [5] english oak
  [6] european ash
  [7] european beech
  [8] european larch
  [9] japanese larch
  [10] linden
  [11] norway spruce
  [12] poplar
  [13] red oak
  [14] scots pine
  [15] sessile oak
  [16] silver fir
  [17] sycamore maple
  [18] weymouth pine

▶ 5x5_test_data_alder  (193 trees)
  → 193 samples saved so far

▶ 5x5_test_data_birch  (165 trees)
  → 358 samples saved so far

▶ 5x5_test_data_blackpine  (2 trees)
  → 360 samples saved so far

▶ 5x5_test_data_cherry  (14 trees)
  → 374 samples saved so far

▶ 5x5_test_data_douglasfir  (420 trees)
  → 794 samples saved so far

▶ 5x5_test_data_englishoak  (274 trees)
  → 1068 samples saved so far

▶ 5x5_test_data_europeanash  (48 trees)
  → 1116 samples saved so far

▶ 5x5_test_data_europeanbeech  (171 trees)
  → 1287 samples saved so far

▶ 5x5_test_data_europeanlarch  (117 trees)
  → 1404 samples saved so far

▶ 5x5_test_data_japanesel

In [19]:
import geopandas as gpd
import numpy as np
import os, json, pickle, ast
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
GEOJSON_FOLDER = "/home/group4/project/test_data" #( 5x5)
OUTPUT_FOLDER  = "/home/group4/Transformers_Nathalia/project/Output_data/test_allbands"
LABEL_LEVEL    = "l3_species"   # 19 classes — change to "l2_genus" or "l1_leaf_types" if needed
# ────────────────────────────────────────────────────────────────────────────

BANDS = ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'EVI', 'EVI2', 'NDVI', 'NDWI', 'SAVI']

# (Make sure 'geometry' is removed as we discussed previously)
N_DATES  = 8
N_PIXELS = 25   # 5x5 patch, flattened

# Actual acquisition dates from your GEE script
DATES = {
    "0": "20220301",
    "1": "20220401",
    "2": "20220501",
    "3": "20220601",
    "4": "20220701",
    "5": "20220801",
    "6": "20220901",
    "7": "20221001",
}

def get_col(band, t):
    return band if t == 0 else f"{band}_{t}"

def parse_cell(val):
    """Parse nested list cell → flat float32 array of length N_PIXELS."""
    if val is None:
        return None
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except:
            return None
    arr = np.array(val, dtype=np.float32).flatten()  # (5,5) → (25,)
    if len(arr) != N_PIXELS:
        return None  # unexpected shape, skip
    return arr

def row_to_tcs(row):
    """
    Convert one GDF row → array (T=8, C=15, S=25).
    Also returns a boolean valid mask of shape (T,).
    """
    out   = np.zeros((N_DATES, len(BANDS), N_PIXELS), dtype=np.float32)
    valid = np.ones(N_DATES, dtype=bool)

    for t in range(N_DATES):
        for c, band in enumerate(BANDS):
            arr = parse_cell(row[get_col(band, t)])
            if arr is None:
                valid[t] = False
                out[t, c, :] = 0.0   # fill missing date with zeros
            else:
                out[t, c, :] = arr

    return out, valid

# ── MAIN BUILD ───────────────────────────────────────────────────────────────
def build_dataset(geojson_folder, output_folder, label_level=LABEL_LEVEL):
    out = Path(output_folder)
    (out / "DATA").mkdir(parents=True, exist_ok=True)
    (out / "META").mkdir(parents=True, exist_ok=True)

    files = sorted([f for f in os.listdir(geojson_folder) if f.endswith(".geojson")])

    # ── Build label map from all files ──────────────────────────────────────
    label_set = set()
    for fname in files:
        gdf = gpd.read_file(os.path.join(geojson_folder, fname))
        label_set.update(gdf[label_level].dropna().unique())
    label_map = {name: i for i, name in enumerate(sorted(label_set))}
    print(f"Found {len(label_map)} classes:")
    for name, idx in label_map.items():
        print(f"  [{idx}] {name}")

    # ── Process each file ───────────────────────────────────────────────────
    labels_dict = {}
    all_arrays  = []
    sample_id   = 0
    skipped     = 0

    for fname in files:
        gdf = gpd.read_file(os.path.join(geojson_folder, fname))
        species = fname.replace(".geojson", "")
        print(f"\n▶ {species}  ({len(gdf)} trees)")

        for _, row in gdf.iterrows():
            label = row.get(label_level)
            if label not in label_map:
                skipped += 1
                continue

            arr, valid = row_to_tcs(row)

            # Skip if more than 2 dates are missing (out of 8)
            if valid.sum() < 6:
                skipped += 1
                continue

            uid = f"sample_{sample_id:06d}"
            np.save(out / "DATA" / f"{uid}.npy", arr)
            labels_dict[uid] = label_map[label]
            all_arrays.append(arr)
            sample_id += 1

        print(f"  → {sample_id} samples saved so far")

    print(f"\n✓ Total samples : {sample_id}")
    print(f"  Skipped        : {skipped}")
    print(f"  Array shape    : T={N_DATES}, C={len(BANDS)}, S={N_PIXELS}")

    # ── labels.json ─────────────────────────────────────────────────────────
    with open(out / "META" / "labels.json", "w") as f:
        json.dump({label_level: labels_dict}, f, indent=2)

    # ── class_map.json (human-readable, for your reference) ─────────────────
    with open(out / "META" / "class_map.json", "w") as f:
        json.dump({v: k for k, v in label_map.items()}, f, indent=2)  # idx → name

    # ── dates.json ──────────────────────────────────────────────────────────
    with open(out / "META" / "dates.json", "w") as f:
        json.dump(DATES, f, indent=2)

    # ── normalisation_values.pkl (channel-wise mean/std) ────────────────────
    big   = np.stack(all_arrays, axis=0)          # (N, T, C, S)
    means = big.mean(axis=(0, 1, 3))              # (C,)
    stds  = big.std(axis=(0, 1, 3))               # (C,)
    stds[stds == 0] = 1e-6
    with open(out / "normalisation_values.pkl", "wb") as f:
        pickle.dump((means, stds), f)

    print(f"✓ Normalisation: means={np.round(means,4)}, stds={np.round(stds,4)}")
    print(f"\n✓ Dataset ready at: {output_folder}")
    print(f"  class_map saved to META/class_map.json — keep this for decoding predictions!")

build_dataset(GEOJSON_FOLDER, OUTPUT_FOLDER)

Found 19 classes:
  [0] alder
  [1] birch
  [2] black pine
  [3] cherry
  [4] douglas fir
  [5] english oak
  [6] european ash
  [7] european beech
  [8] european larch
  [9] japanese larch
  [10] linden
  [11] norway spruce
  [12] poplar
  [13] red oak
  [14] scots pine
  [15] sessile oak
  [16] silver fir
  [17] sycamore maple
  [18] weymouth pine

▶ 5x5_test_data_alder  (193 trees)
  → 193 samples saved so far

▶ 5x5_test_data_birch  (165 trees)
  → 358 samples saved so far

▶ 5x5_test_data_blackpine  (2 trees)
  → 360 samples saved so far

▶ 5x5_test_data_cherry  (14 trees)
  → 374 samples saved so far

▶ 5x5_test_data_douglasfir  (420 trees)
  → 794 samples saved so far

▶ 5x5_test_data_englishoak  (274 trees)
  → 1068 samples saved so far

▶ 5x5_test_data_europeanash  (48 trees)
  → 1116 samples saved so far

▶ 5x5_test_data_europeanbeech  (171 trees)
  → 1287 samples saved so far

▶ 5x5_test_data_europeanlarch  (117 trees)
  → 1404 samples saved so far

▶ 5x5_test_data_japanesel

In [20]:
import os
import geopandas as gpd

# Set this to the same folder you are using in your main script
GEOJSON_FOLDER = "/home/group4/project/5_5_DEM"

# List all files in the directory
files = [f for f in os.listdir(GEOJSON_FOLDER) if f.endswith('.geojson')]

print(f"Checking {len(files)} files...\n" + "-"*50)

for fname in files:
    file_path = os.path.join(GEOJSON_FOLDER, fname)
    try:
        gdf = gpd.read_file(file_path)
        print(f"File: {fname}")
        print(f"Columns: {list(gdf.columns)}")
        print("-" * 50)
    except Exception as e:
        print(f"Could not read {fname}: {e}")

Checking 33 files...
--------------------------------------------------
File: 5x5_DEM_test_data_poplar.geojson
Columns: ['id', 'B11', 'B11_1', 'B11_2', 'B11_3', 'B11_4', 'B11_5', 'B11_6', 'B11_7', 'B12', 'B12_1', 'B12_2', 'B12_3', 'B12_4', 'B12_5', 'B12_6', 'B12_7', 'B2', 'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5', 'B2_6', 'B2_7', 'B3', 'B3_1', 'B3_2', 'B3_3', 'B3_4', 'B3_5', 'B3_6', 'B3_7', 'B4', 'B4_1', 'B4_2', 'B4_3', 'B4_4', 'B4_5', 'B4_6', 'B4_7', 'B5', 'B5_1', 'B5_2', 'B5_3', 'B5_4', 'B5_5', 'B5_6', 'B5_7', 'B6', 'B6_1', 'B6_2', 'B6_3', 'B6_4', 'B6_5', 'B6_6', 'B6_7', 'B7', 'B7_1', 'B7_2', 'B7_3', 'B7_4', 'B7_5', 'B7_6', 'B7_7', 'B8', 'B8A', 'B8A_1', 'B8A_2', 'B8A_3', 'B8A_4', 'B8A_5', 'B8A_6', 'B8A_7', 'B8_1', 'B8_2', 'B8_3', 'B8_4', 'B8_5', 'B8_6', 'B8_7', 'EVI', 'EVI2', 'EVI2_1', 'EVI2_2', 'EVI2_3', 'EVI2_4', 'EVI2_5', 'EVI2_6', 'EVI2_7', 'EVI_1', 'EVI_2', 'EVI_3', 'EVI_4', 'EVI_5', 'EVI_6', 'EVI_7', 'NDVI', 'NDVI_1', 'NDVI_2', 'NDVI_3', 'NDVI_4', 'NDVI_5', 'NDVI_6', 'NDVI_7', 'ND

KeyboardInterrupt: 

In [13]:
import os
import geopandas as gpd

# Set this to the same folder you are using in your main script
GEOJSON_FOLDER = "/home/group4/project/test_data"

# List all files in the directory
files = [f for f in os.listdir(GEOJSON_FOLDER) if f.endswith('.geojson')]

print(f"Checking {len(files)} files...\n" + "-"*50)

for fname in files:
    file_path = os.path.join(GEOJSON_FOLDER, fname)
    try:
        gdf = gpd.read_file(file_path)
        print(f"File: {fname}")
        print(f"Columns: {list(gdf.columns)}")
        print("-" * 50)
    except Exception as e:
        print(f"Could not read {fname}: {e}")

Checking 19 files...
--------------------------------------------------
File: 5x5_test_data_norwayspruce.geojson
Columns: ['id', 'B11', 'B11_1', 'B11_2', 'B11_3', 'B11_4', 'B11_5', 'B11_6', 'B11_7', 'B12', 'B12_1', 'B12_2', 'B12_3', 'B12_4', 'B12_5', 'B12_6', 'B12_7', 'B2', 'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5', 'B2_6', 'B2_7', 'B3', 'B3_1', 'B3_2', 'B3_3', 'B3_4', 'B3_5', 'B3_6', 'B3_7', 'B4', 'B4_1', 'B4_2', 'B4_3', 'B4_4', 'B4_5', 'B4_6', 'B4_7', 'B5', 'B5_1', 'B5_2', 'B5_3', 'B5_4', 'B5_5', 'B5_6', 'B5_7', 'B6', 'B6_1', 'B6_2', 'B6_3', 'B6_4', 'B6_5', 'B6_6', 'B6_7', 'B7', 'B7_1', 'B7_2', 'B7_3', 'B7_4', 'B7_5', 'B7_6', 'B7_7', 'B8', 'B8A', 'B8A_1', 'B8A_2', 'B8A_3', 'B8A_4', 'B8A_5', 'B8A_6', 'B8A_7', 'B8_1', 'B8_2', 'B8_3', 'B8_4', 'B8_5', 'B8_6', 'B8_7', 'EVI', 'EVI2', 'EVI2_1', 'EVI2_2', 'EVI2_3', 'EVI2_4', 'EVI2_5', 'EVI2_6', 'EVI2_7', 'EVI_1', 'EVI_2', 'EVI_3', 'EVI_4', 'EVI_5', 'EVI_6', 'EVI_7', 'NDVI', 'NDVI_1', 'NDVI_2', 'NDVI_3', 'NDVI_4', 'NDVI_5', 'NDVI_6', 'NDVI_7', '

File: 5x5_test_data_douglasfir.geojson
Columns: ['id', 'B11', 'B11_1', 'B11_2', 'B11_3', 'B11_4', 'B11_5', 'B11_6', 'B11_7', 'B12', 'B12_1', 'B12_2', 'B12_3', 'B12_4', 'B12_5', 'B12_6', 'B12_7', 'B2', 'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5', 'B2_6', 'B2_7', 'B3', 'B3_1', 'B3_2', 'B3_3', 'B3_4', 'B3_5', 'B3_6', 'B3_7', 'B4', 'B4_1', 'B4_2', 'B4_3', 'B4_4', 'B4_5', 'B4_6', 'B4_7', 'B5', 'B5_1', 'B5_2', 'B5_3', 'B5_4', 'B5_5', 'B5_6', 'B5_7', 'B6', 'B6_1', 'B6_2', 'B6_3', 'B6_4', 'B6_5', 'B6_6', 'B6_7', 'B7', 'B7_1', 'B7_2', 'B7_3', 'B7_4', 'B7_5', 'B7_6', 'B7_7', 'B8', 'B8A', 'B8A_1', 'B8A_2', 'B8A_3', 'B8A_4', 'B8A_5', 'B8A_6', 'B8A_7', 'B8_1', 'B8_2', 'B8_3', 'B8_4', 'B8_5', 'B8_6', 'B8_7', 'EVI', 'EVI2', 'EVI2_1', 'EVI2_2', 'EVI2_3', 'EVI2_4', 'EVI2_5', 'EVI2_6', 'EVI2_7', 'EVI_1', 'EVI_2', 'EVI_3', 'EVI_4', 'EVI_5', 'EVI_6', 'EVI_7', 'NDVI', 'NDVI_1', 'NDVI_2', 'NDVI_3', 'NDVI_4', 'NDVI_5', 'NDVI_6', 'NDVI_7', 'NDWI', 'NDWI_1', 'NDWI_2', 'NDWI_3', 'NDWI_4', 'NDWI_5', 'NDWI_6', 'NDWI_7

File: 5x5_test_data_poplar.geojson
Columns: ['id', 'B11', 'B11_1', 'B11_2', 'B11_3', 'B11_4', 'B11_5', 'B11_6', 'B11_7', 'B12', 'B12_1', 'B12_2', 'B12_3', 'B12_4', 'B12_5', 'B12_6', 'B12_7', 'B2', 'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5', 'B2_6', 'B2_7', 'B3', 'B3_1', 'B3_2', 'B3_3', 'B3_4', 'B3_5', 'B3_6', 'B3_7', 'B4', 'B4_1', 'B4_2', 'B4_3', 'B4_4', 'B4_5', 'B4_6', 'B4_7', 'B5', 'B5_1', 'B5_2', 'B5_3', 'B5_4', 'B5_5', 'B5_6', 'B5_7', 'B6', 'B6_1', 'B6_2', 'B6_3', 'B6_4', 'B6_5', 'B6_6', 'B6_7', 'B7', 'B7_1', 'B7_2', 'B7_3', 'B7_4', 'B7_5', 'B7_6', 'B7_7', 'B8', 'B8A', 'B8A_1', 'B8A_2', 'B8A_3', 'B8A_4', 'B8A_5', 'B8A_6', 'B8A_7', 'B8_1', 'B8_2', 'B8_3', 'B8_4', 'B8_5', 'B8_6', 'B8_7', 'EVI', 'EVI2', 'EVI2_1', 'EVI2_2', 'EVI2_3', 'EVI2_4', 'EVI2_5', 'EVI2_6', 'EVI2_7', 'EVI_1', 'EVI_2', 'EVI_3', 'EVI_4', 'EVI_5', 'EVI_6', 'EVI_7', 'NDVI', 'NDVI_1', 'NDVI_2', 'NDVI_3', 'NDVI_4', 'NDVI_5', 'NDVI_6', 'NDVI_7', 'NDWI', 'NDWI_1', 'NDWI_2', 'NDWI_3', 'NDWI_4', 'NDWI_5', 'NDWI_6', 'NDWI_7', '

In [11]:
import os
import numpy as np

# [Your configs list goes here]
configs = [
    {"model_name": "PseTae_5x5_test", "folder": "/home/group4/Transformers_Nathalia/project/Output_data/test_vegetation", "npixel": 25, "n_classes": 19, "bands": ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12',
         'EVI', 'EVI2', 'NDVI', 'NDWI', 'SAVI']},
]

print(f"{'MODEL NAME':<45} | {'EXPECTED':<10} | {'ACTUAL':<10} | {'STATUS'}")
print("-" * 85)

for config in configs:
    folder = config['folder']
    data_path = os.path.join(folder, "DATA")
    expected_bands = len(config['bands'])
    
    # 1. Check if folder exists
    if not os.path.exists(data_path):
        print(f"{config['model_name']:<45} | {expected_bands:<10} | {'MISSING':<10} | ❌ Path not found")
        continue
    
    # 2. Get first file
    files = [f for f in os.listdir(data_path) if f.endswith('.npy')]
    if not files:
        print(f"{config['model_name']:<45} | {expected_bands:<10} | {'EMPTY':<10} | ❌ No data")
        continue
        
    # 3. Load data and check shape
    sample = np.load(os.path.join(data_path, files[0]))
    # Assuming shape (Time, Bands, Pixels)
    actual_bands = sample.shape[1]
    
    status = "✅ MATCH" if expected_bands == actual_bands else "⚠️ MISMATCH"
    print(f"{config['model_name']:<45} | {expected_bands:<10} | {actual_bands:<10} | {status}")

MODEL NAME                                    | EXPECTED   | ACTUAL     | STATUS
-------------------------------------------------------------------------------------
PseTae_5x5_test                               | 15         | 5          | ⚠️ MISMATCH


In [21]:
import os
import geopandas as gpd

GEOJSON_FOLDER = "/home/group4/project/5_5_DEM"

# Define the columns to ignore so they aren't parsed as dates
EXCLUDE_COLS = ['id', 'geometry', 'l1_leaf_types', 'l2_genus', 'l3_species']

all_found_indices = set()

# Grab just one file to check the structure (assuming all files follow the same format)
files = [f for f in os.listdir(GEOJSON_FOLDER) if f.endswith('.geojson')]

if not files:
    print("No files found.")
else:
    # We only need to check the first file to identify the structure of your dataset
    sample_file = files[0]
    gdf = gpd.read_file(os.path.join(GEOJSON_FOLDER, sample_file))
    
    print(f"Analyzing file: {sample_file}")
    
    for col in gdf.columns:
        if col in EXCLUDE_COLS:
            continue
            
        # Logic: 
        # If no underscore, it is index 0 (e.g., 'B11')
        # If underscore exists, the part after the last underscore is the index (e.g., 'B11_1')
        if "_" in col:
            try:
                date_idx = int(col.split('_')[-1])
                all_found_indices.add(date_idx)
            except ValueError:
                # This handles columns that might contain underscores but aren't date indices
                continue
        else:
            # If there is no underscore, we assume it belongs to index 0
            all_found_indices.add(0)

    print("-" * 50)
    print(f"Date Indices detected in dataset: {sorted(list(all_found_indices))}")
    
    # Optional: Map to your specific dates
    DATES_MAP = {
        0: "20220301", 1: "20220401", 2: "20220501", 3: "20220601",
        4: "20220701", 5: "20220801", 6: "20220901", 7: "20221001"
    }
    
    print("\nCorresponding Calendar Dates:")
    for idx in sorted(list(all_found_indices)):
        print(f"Index {idx}: {DATES_MAP.get(idx, 'Unknown')}")

Analyzing file: 5x5_DEM_test_data_poplar.geojson
--------------------------------------------------
Date Indices detected in dataset: [0, 1, 2, 3, 4, 5, 6, 7]

Corresponding Calendar Dates:
Index 0: 20220301
Index 1: 20220401
Index 2: 20220501
Index 3: 20220601
Index 4: 20220701
Index 5: 20220801
Index 6: 20220901
Index 7: 20221001
